# GAN-PRNG 随机数生成 Notebook

使用训练好的生成器模型生成随机数并导出为二进制文件

## 使用说明
1. 加载训练好的模型检查点
2. 配置生成参数
3. 生成随机比特
4. 评估随机性质量
5. 导出为二进制文件

### 1. 导入必要的库

In [21]:
import torch
import torch.nn as nn
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime

# 导入模块化组件
from evaluator import AnalyzerRuns, evaluate_randomness
from utils import (
    load_generator,
    generate_random_bits,
    bits_to_bytes,
    save_to_binary,
    generate_speed
)

### 2. 配置参数

In [22]:
# ========== 模型配置 ==========
CHECKPOINT_PATH = './checkpoints/GAN_train_20260709_194644/生成器_轮次_200000.pth'
# TODO: 填写检查点文件路径，例如：'./checkpoints/GAN_训练_20260311_120000/生成器_轮次_43000.pth'
MODEL_TYPE = 2         # 0=基础模型，1=双生成器模型
SEQ_LEN = 512          # 序列长度（必须与训练时一致）
Z_DIM = 64             # 噪声维度（必须与训练时一致）

# ========== 生成配置 ==========
TOTAL_BITS = 1000000*1000   # 需要生成的总比特数
BATCH_SIZE = 512       # 生成时的批大小
USE_CUDA = True        # 是否使用 CUDA

# ========== 输出配置 ==========
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = f'./output/random_{TIMESTAMP}.bin'  # 输出文件路径

# ========== 其他配置 ==========
BIT_MODE = 1           # 0=传统模式，1=比特展开模式
EVALUATE = True        # 是否进行随机性评估

# 设置设备
DEVICE = "cuda" if (torch.cuda.is_available() and USE_CUDA) else "cpu"

print(f"使用设备：{DEVICE}")
print(f"模型类型：{'基础模型' if MODEL_TYPE == 0 else '双生成器模型'}")
print(f"目标比特数：{TOTAL_BITS:,} ({TOTAL_BITS / (1024*1024):.2f} MB)")
print(f"输出文件：{OUTPUT_PATH}")

使用设备：cuda
模型类型：双生成器模型
目标比特数：1,000,000,000 (953.67 MB)
输出文件：./output/random_20260709_224742.bin


### 3. 加载训练好的模型

In [23]:
# 检查检查点文件是否存在
if not CHECKPOINT_PATH:
    print("错误：请设置 CHECKPOINT_PATH 为有效的检查点文件路径！")
    print("\n可用的检查点文件：")
    
    # 自动查找可用的检查点
    checkpoint_dir = './checkpoints'
    if os.path.exists(checkpoint_dir):
        for root, dirs, files in os.walk(checkpoint_dir):
            for file in files:
                if file.endswith('.pth'):
                    full_path = os.path.join(root, file)
                    print(f"  - {full_path}")
    else:
        print(f"  目录 {checkpoint_dir} 不存在")
    
    print("\n请先在 train.ipynb 中训练模型，或手动指定检查点路径")
else:
    # 加载生成器
    if os.path.exists(CHECKPOINT_PATH):
        generator = load_generator(
            checkpoint_path=CHECKPOINT_PATH,
            model_type=MODEL_TYPE,
            z_dim=Z_DIM,
            seq_len=SEQ_LEN,
            device=DEVICE
        )
        print("\n模型加载成功！")
        print("\n生成器结构:")
        print(generator)
    else:
        print(f"\n错误：检查点文件不存在：{CHECKPOINT_PATH}")
        print("请检查路径是否正确")

已加载检查点：./checkpoints/GAN_train_20260709_194644/生成器_轮次_200000.pth
模型类型：大规模基础模型

模型加载成功！

生成器结构:
GAN_g(
  (gen1): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): LeakyReLU(negative_slope=0.2, inplace=True)
    (4): HardMod()
    (5): Linear(in_features=512, out_features=768, bias=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Linear(in_features=768, out_features=1024, bias=True)
    (8): LeakyReLU(negative_slope=0.2, inplace=True)
    (9): Linear(in_features=1024, out_features=512, bias=True)
    (10): HardMod()
  )
)


### 4. 生成随机比特

In [24]:
import time

if 'generator' in locals():
    # 生成随机比特（含速度计时）
    print("=" * 50)
    print("开始生成随机比特...")
    t_start = time.time()

    bits = generate_random_bits(
        generator=generator,
        total_bits=TOTAL_BITS,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        z_dim=Z_DIM,
        bit_mode=1
    )



开始生成随机比特...
正在生成 1,000,000,000 个随机比特...
torch.Size([262144])
tensor(27.0312, device='cuda:0')
进度：100.0% (1,000,000,000/1,000,000,000)
[0 0 0 1 1 0 1 1] 477


### 5. 导出为二进制文件

In [25]:
if 'bits' in locals():
    # 转换为字节
    print("正在转换为字节格式...")
    byte_data = bits_to_bytes(bits)
    
    # 保存到二进制文件
    save_to_binary(byte_data, OUTPUT_PATH)
    
    print(f"\n生成完成！")
    print(f"输出文件：{OUTPUT_PATH}")
    print(f"总比特数：{len(bits):,}")
    print(f"字节数：{len(byte_data):,}")
    print(f"文件大小：{len(byte_data) / (1024*1024):.2f} MB")

正在转换为字节格式...
已保存至：./output/random_20260709_224742.bin
文件大小：119.21 MB

生成完成！
输出文件：./output/random_20260709_224742.bin
总比特数：1,000,000,000
字节数：125,000,000
文件大小：119.21 MB


### 6. 测试生成速度

In [26]:
  result = generate_speed(
      generator=generator,
      total_bits=100_000_000,   # 1MB
      device='cuda',
      batch_size=BATCH_SIZE,
      z_dim=Z_DIM,
      bit_mode=1
  )

=== G(z) 纯推理速度 ===
设备: cuda | batch: 512 | seq_len: 512
单次 forward:       0.7409 ms
吞吐量:             691,065 samples/s | 2830.60 Mbps | 2.831 Gbps
生成 100,000,000 bits 等效耗时: 35.33 ms
(不含噪声生成、不含比特转换、不含 CPU 传输)


### 8. 统计信息汇总

In [27]:
if 'bits' in locals():
    print("\n" + "="*60)
    print("生成任务汇总")
    print("="*60)
    print(f"生成时间：{TIMESTAMP}")
    print(f"模型检查点：{CHECKPOINT_PATH}")
    print(f"模型类型：{'基础模型' if MODEL_TYPE == 0 else '双生成器模型'}")
    print(f"设备：{DEVICE}")
    print(f"\n生成参数:")
    print(f"  - 总比特数：{TOTAL_BITS:,}")
    print(f"  - 批大小：{BATCH_SIZE}")
    print(f"  - 噪声维度：{Z_DIM}")
    print(f"  - 序列长度：{SEQ_LEN}")
    print(f"\n输出文件:")
    print(f"  - 路径：{OUTPUT_PATH}")
    print(f"  - 大小：{os.path.getsize(OUTPUT_PATH) / (1024*1024):.2f} MB" if os.path.exists(OUTPUT_PATH) else "  - 未保存")
    print(f"\n随机性质量:")
    if 'metrics' in locals():
        print(f"  - 1 占比：{metrics['1 占比']:.4f} (理想值：0.5)")
        print(f"  - 自相关性：{metrics['自相关性']:.4f} (理想值：0)")
        print(f"  - Runs_P 值：{metrics['Runs_P 值']:.6f} (理想值：>0.01)")
    print("="*60)


生成任务汇总
生成时间：20260709_224742
模型检查点：./checkpoints/GAN_train_20260709_194644/生成器_轮次_200000.pth
模型类型：双生成器模型
设备：cuda

生成参数:
  - 总比特数：1,000,000,000
  - 批大小：512
  - 噪声维度：64
  - 序列长度：512

输出文件:
  - 路径：./output/random_20260709_224742.bin
  - 大小：119.21 MB

随机性质量:


### 5. 随机性质量评估

In [28]:
%%script false

if 'bits' in locals() and EVALUATE:
    # 评估随机性
    bits = bits[:1000000]
    
    metrics = evaluate_randomness(bits, bit_mode=0)
    
    # 可视化随机性指标
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # 1. 比特分布
    ax = axes[0]
    ax.bar(['0', '1'], [len(bits) - np.sum(bits), np.sum(bits)], color=['blue', 'red'], alpha=0.7)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Bit Distribution', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 添加理想值参考线
    ideal = len(bits) / 2
    ax.axhline(y=ideal, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Ideal')
    ax.legend()

    
    # 2. 自相关性分析（前 100 个滞后）
    ax = axes[1]
    max_lag = min(50, len(bits) // 10)
    autocorrs = []
    for lag in range(1, max_lag + 1):
        if len(bits) > lag:
            corr = np.corrcoef(bits[:-lag], bits[lag:])[0, 1]
            autocorrs.append(corr if not np.isnan(corr) else 0)
    
    ax.plot(range(1, len(autocorrs) + 1), autocorrs, 'o-', markersize=3, linewidth=1)
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Lag', fontsize=12)
    ax.set_ylabel('Autocorrelation', fontsize=12)
    ax.set_title('Autocorrelation vs Lag', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    # 3. Runs 分布
    ax = axes[2]
    # 计算 runs（连续相同比特的段）
    runs = []
    current_run = 1
    for i in range(1, min(10000, len(bits))):
        if bits[i] == bits[i-1]:
            current_run += 1
        else:
            runs.append(current_run)
            current_run = 1
    runs.append(current_run)
    
    # 绘制 runs 长度分布
    max_run_len = min(20, max(runs))
    run_counts = [runs.count(i) for i in range(1, max_run_len + 1)]
    ax.bar(range(1, max_run_len + 1), run_counts, alpha=0.7, color='green')
    ax.set_xlabel('Run Length', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Distribution of Run Lengths', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # 保存图表
    plot_path = f'./output/randomness_analysis_{TIMESTAMP}.png'
    os.makedirs(os.path.dirname(plot_path), exist_ok=True)
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n随机性分析图表已保存至：{plot_path}")

Couldn't find program: 'false'


### 7. 批量生成多个文件（可选）

In [29]:
%%script false
# 如果需要批量生成多个随机数文件，可以运行以下代码
'''
if 'generator' in locals():
    # 批量生成配置
    NUM_FILES = 1       # 要生成的文件数量
    BITS_PER_FILE = 1048576  # 每个文件的比特数

    print(f"准备批量生成 {NUM_FILES} 个文件，每个文件 {BITS_PER_FILE:,} 比特...")

    output_dir = f'./output/batch_{TIMESTAMP}'
    os.makedirs(output_dir, exist_ok=True)

    for i in range(NUM_FILES):
        print(f"\n生成第 {i+1}/{NUM_FILES} 个文件...")

        # 生成随机比特
        bits_batch = generate_random_bits(
            generator=generator,
            total_bits=BITS_PER_FILE,
            device=DEVICE,
            batch_size=BATCH_SIZE,
            z_dim=Z_DIM
        )

        # 保存
        file_path = os.path.join(output_dir, f'random_{i+1:03d}.bin')
        byte_data_batch = bits_to_bytes(bits_batch)
        save_to_binary(byte_data_batch, file_path)

    print(f"\n批量生成完成！")
    print(f"所有文件已保存至：{output_dir}")
'''

Couldn't find program: 'false'


### 9. 验证生成的文件（可选）

In [30]:
%%script false
# 读取并验证生成的二进制文件
if 'OUTPUT_PATH' in locals() and os.path.exists(OUTPUT_PATH):
    print(f"验证文件：{OUTPUT_PATH}\n")
    
    # 读取文件
    with open(OUTPUT_PATH, 'rb') as f:
        file_data = f.read()
    
    # 转换为比特
    file_bits = np.unpackbits(np.frombuffer(file_data, dtype=np.uint8))
    
    # 计算统计信息
    file_size = len(file_data)
    total_bits = len(file_bits)
    ones_count = np.sum(file_bits)
    ones_ratio = ones_count / total_bits
    
    print(f"文件大小：{file_size / (1024*1024):.2f} MB")
    print(f"总比特数：{total_bits:,}")
    print(f"1 的数量：{ones_count:,}")
    print(f"0 的数量：{total_bits - ones_count:,}")
    print(f"1 的比例：{ones_ratio:.4f}")
    
    # 计算文件指标的自相关性
    if len(file_bits) > 1:
        autocorr = np.corrcoef(file_bits[:-1], file_bits[1:])[0, 1]
        print(f"自相关性：{autocorr:.4f}")
    
    print("\n文件验证完成！")

Couldn't find program: 'false'


### 10. 下一步操作

生成完成后，您可以：
1. 使用生成的随机数文件进行各种应用
2. 使用专业统计测试工具（如 NIST STS）进行更严格的测试
3. 调整参数重新生成
4. 返回 `train.ipynb` 继续训练模型以获得更好的质量